In [1]:
# Parameters
BATCH_MODE = "true"


# Duel Verbal : Barbie vs l'Âne de Shrek
## Introduction - barbie schreck

**Navigation** : [Index](../README.md)

Ce notebook couvre les concepts et techniques principaux de barbie schreck. Les objectifs pedagogiques incluent la comprehension des fondamentaux, la mise en pratique via des exercices, et analyse de résultats.

**Prerequis** : notions de base en Python et en algorithmique.


> **Adapté d'une production étudiante EPF** : **Carole & Cléo**. Universalisation : refactor [#890](https://github.com/jsboige/CoursIA/pull/890).


Notebook utilisant **Semantic Kernel** pour organiser un débat contraint entre deux agents — Barbie et l'Âne de Shrek — où chaque réplique de l'Âne est illustrée par une image générée sur l'instance **ComfyUI** de la série.

Le fil conducteur pédagogique en 4 actes :

1. **Contrainte** — un style linguistique (rime, Shakespeare, chanson) est tiré au sort et imposé aux deux agents via leurs instructions système ;
2. **Agents** — deux `ChatCompletionAgent` aux personnalités opposées, chacun sur son propre kernel ;
3. **Plugin** — un `ImagePlugin` branché sur l'Âne illustre chaque réplique via l'API de l'instance ComfyUI ;
4. **Orchestration** — un `AgentGroupChat` + une stratégie de terminaison font tourner le duel jusqu'à `MAX_TURNS`.

Les trois exercices d'origine ont été résolus par un groupe de l'APP3A (PR #21089) : leurs solutions sont intégrées en **exemples guidés 1, 2 et 3**, chacun suivi de la lecture de son résultat et de ses limites. Trois **exercices de remplacement** les prolongent dans la section « 9. Exercices ».


In [2]:
# Import guards - availability flags for external dependencies

try:
    from IPython.display import display, Audio, Image as IPImage
    IPYTHON_AVAILABLE = True
except ImportError:
    IPYTHON_AVAILABLE = False
    print(f'  IPython non disponible - certaines fonctionnalites seront limitees')

try:
    from dotenv import load_dotenv
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False
    print(f'  dotenv non disponible - certaines fonctionnalites seront limitees')

try:
    import semantic_kernel
    SEMANTIC_KERNEL_AVAILABLE = True
except ImportError:
    SEMANTIC_KERNEL_AVAILABLE = False
    print(f'  semantic_kernel non disponible - certaines fonctionnalites seront limitees')


## Configuration initiale

Cette cellule importe toutes les briques nécessaires et prépare l'environnement :

- **`os`, `random`, `logging`** — utilitaires standard (variables d'env, tirage de contrainte, logs) ;
- **`dotenv`** — `load_dotenv()` charge les clés API depuis le fichier `.env` local (voir `OPENAI_API_KEY` utilisée plus bas) ;
- **`semantic_kernel`** — le cœur : `Kernel`, `ChatCompletionAgent`, `AgentGroupChat`, les connecteurs OpenAI (chat + texte-à-image), `kernel_function`, `ChatMessageContent`/`AuthorRole`.

Le `logging.basicConfig(level=logging.INFO)` active les logs INFO qui rendront l'exécution du débat lisible (sélection d'agent, usage API). L'output `Imports et configuration OK` confirme que toutes les dépendances sont présentes sur la machine d'exécution.

La cellule `# Import guards` précédente joue un rôle de pré-vol : elle vérifie la disponibilité de `IPython`, `dotenv` et `semantic_kernel` et définit des flags (`IPYTHON_AVAILABLE`, etc.) — **mais ces gardes ne protègent pas le carnet** : la cellule suivante importe `dotenv` et `semantic_kernel` au niveau module, sans garde, si bien qu'une dépendance manquante interrompt l'exécution à cet import. Les flags documentent l'état de la machine, ils n'arrêtent rien.

In [3]:
import os
import random
import logging
from pathlib import Path
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.functions import kernel_function, KernelArguments
from semantic_kernel.contents import ChatMessageContent, AuthorRole


def _charger_env_genai():
    """Charge le .env de la serie GenAI en remontant depuis le dossier du carnet.

    load_dotenv() nu chercherait l'emplacement de l'appelant, pas le dossier du
    carnet : sous papermill comme dans un terminal, la remontee explicite depuis
    le repertoire courant est la seule forme fiable.
    """
    # Remontee d'au plus 6 niveaux : le carnet vit a 3 niveaux sous la racine
    # de la serie ; la marge couvre les executions depuis un autre cwd.
    courant = Path.cwd()
    for _ in range(6):
        env = courant / ".env"
        if env.exists():
            load_dotenv(dotenv_path=env, override=True)
            return f"{env.parent.name}/{env.name}"
        courant = courant.parent
    return None


# Point d'entree unique pour les cles de la serie (endpoint du modele de chat).
CHEMIN_ENV = _charger_env_genai()
print("Env de la série chargé :", CHEMIN_ENV or "introuvable")

# Journalisation shared par le duel et les strategies de terminaison.
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('BarbieVsAne')
print("Imports et configuration OK")


Env de la série chargé : GenAI/.env
Imports et configuration OK


## Définition des contraintes linguistiques

Le débat ne gagne en intérêt que si les répliques obéissent à une règle. La liste `CONTRAINTES` définit **trois styles possibles**, chacun sous la forme d'un tuple `(nom, description)` :

| Nom | Règle |
|---|---|
| `Rime` | Chaque réplique contient une rime parfaite |
| `Shakespeare` | Imiter le style théâtral de Shakespeare |
| `Chanson` | Répondre sur l'air d'I'm a Believer |

`random.choice` tire une contrainte au sort **à chaque exécution** : le tirage n'est donc pas reproductible — la contrainte effective est celle que la cellule suivante affiche, et les répliques du débat la suivent. (Dans la sortie committée, c'est **Chanson** — « Répondre sur l'air de 'I'm a Believer' » — qui a été tirée.) La structure en tuples `(nom, description)` est celle que l'Exercice 1 demande d'étendre : la description est interpolée dans les instructions des agents, donc toute nouvelle entrée de la liste se répercute automatiquement sur le comportement des deux duellistes.

In [4]:
# Contraintes declaratives du duel : un nom court et une phrase d'instruction
# qui sera injectee dans les instructions des agents (section suivante).
CONTRAINTES = [
    ("Rime", "Chaque réplique doit contenir une rime parfaite"),
    ("Shakespeare", "Imiter le style théâtral de Shakespeare"),
    ("Chanson", "Répondre sur l'air de 'I'm a Believer'")
]

# Tirage au sort : chaque execution impose une contrainte differente au duel.
contrainte_choisie = random.choice(CONTRAINTES)
print(f"Contrainte choisie : {contrainte_choisie[0]}")
print("Contraintes de debate definies")


Contrainte choisie : Rime
Contraintes de debate definies


### Exemple guidé 1 — Ajout de contraintes linguistiques personnalisées

*Contribution étudiante du groupe de la PR #21089, intégrée comme exemple guidé.*

La liste `CONTRAINTES` contient trois styles (Rime, Shakespeare, Chanson). L'exemple ajoute trois entrées au format `(nom, description)`. Comme la description est interpolée dans les instructions système des deux agents, chaque entrée ajoutée change leur comportement sans toucher au reste du montage.

**Ce que fait le code ci-dessous** :

- `# Étape 1` — définir les nouvelles contraintes dans `NOUVELLES_CONTRAINTES` (Haiku, Code Python, Pirate) ;
- `# Étape 2` — les rattacher à `CONTRAINTES` avec `extend`, puis afficher la liste complète.

L'indice de l'énoncé d'origine est conservé : respecter le format existant `("Nom", "Description de la règle")`.


In [5]:
# Exemple guidé 1 — Ajout de contraintes linguistiques personnalisées.
# Contribution étudiante du groupe de la PR #21089, intégrée comme exemple guidé.

# Etape 1 : definir les nouvelles contraintes (format : (nom, description))
NOUVELLES_CONTRAINTES = [
    ("Haiku", "Chaque réplique doit être un haïku japonais : trois lignes de 5, 7 et 5 syllabes"),
    ("Code Python", "Répondre uniquement en syntaxe Python (print, def, if/else, boucles...)"),
    ("Pirate", "Parler comme un pirate : 'Arrr', 'matelot', 'par tous les kraken'..."),
]

# Etape 2 : les ajouter a CONTRAINTES
CONTRAINTES.extend(NOUVELLES_CONTRAINTES)

print(f"{len(CONTRAINTES)} contraintes disponibles :")
for nom, description in CONTRAINTES:
    print(f"  - {nom} : {description}")


6 contraintes disponibles :
  - Rime : Chaque réplique doit contenir une rime parfaite
  - Shakespeare : Imiter le style théâtral de Shakespeare
  - Chanson : Répondre sur l'air de 'I'm a Believer'
  - Haiku : Chaque réplique doit être un haïku japonais : trois lignes de 5, 7 et 5 syllabes
  - Code Python : Répondre uniquement en syntaxe Python (print, def, if/else, boucles...)
  - Pirate : Parler comme un pirate : 'Arrr', 'matelot', 'par tous les kraken'...


### Lecture du résultat — exemple guidé 1

La liste passe de 3 à 6 contraintes, et les trois nouvelles sont affichées avec leur description : le format `(nom, description)` est bien celui que les instructions système attendent.

**Limite de l'exemple, à connaître avant de s'en servir.** La contrainte du duel a été tirée au sort **avant** cet ajout (`contrainte_choisie = random.choice(CONTRAINTES)`, cellule de la liste d'origine). Les trois contraintes neuves existent donc dans `CONTRAINTES`, mais le duel qui suit joue encore celle qui a été tirée. Pour qu'une contrainte neuve soit réellement jouée, il faut **tirer après avoir étendu la liste** — c'est précisément ce que fait l'exercice 1 de remplacement, par une autre voie.


## Création du kernel

`create_kernel()` instancie un `Kernel` Semantic Kernel et y enregistre le service de conversation, identifié par son `service_id` :

- **`chat`** : `OpenAIChatCompletion` — le modèle de conversation (id lu depuis `OPENAI_CHAT_MODEL_ID`) ;
- les clés sont lues depuis l'environnement (`os.getenv`), jamais écrites dans le notebook.

Le pattern `service_id` est central : c'est lui qui permet aux agents de retrouver le bon service, même quand un kernel en porte plusieurs. La fonction est appelée **deux fois** (un kernel par agent) : chaque duelliste dispose de sa propre pile de services, ce qui isole leurs appels.

Les images ne passent pas par un service du kernel : elles sont produites par l'**instance ComfyUI de la série**, appelée en HTTP par le plugin de la section suivante. L'URL et le jeton de cette instance vivent dans le `.env` de la série — jamais dans une cellule.


In [6]:
def create_kernel():
    # Fabrique de kernel : chaque agent recoit le sien (section Agent configuration),
    # tous pointent vers le meme service OpenAI-compatible de la serie.
    kernel = Kernel()
    kernel.add_service(OpenAIChatCompletion(
        service_id="chat",
        ai_model_id=os.getenv("OPENAI_CHAT_MODEL_ID"),
        api_key=os.getenv("OPENAI_API_KEY")
    ))
    return kernel

print("Fonction create_kernel définie")


Fonction create_kernel définie


## Plugin de génération d'images

`ImagePlugin` encapsule l'appel à l'instance **ComfyUI** de la série sous forme de **fonction invocable** Semantic Kernel. Le décorateur `@kernel_function(name="generate_image", description="...")` expose la méthode `generate` comme une capacité que le kernel peut invoquer par nom.

Détails de l'implémentation :

- **transport et authentification** : le client partagé `shared/helpers/comfyui_client.py` lit `COMFYUI_API_URL` et le jeton dans le `.env` de la série — le jeton n'apparaît **jamais** dans une cellule ;
- **graphe** : le workflow `qwen_image_2_1_text2image.json` (dossier `scripts/genai-stack/workflows/qwen_image_2_1/`) est chargé puis converti du format *UI* au format *API*. C'est ce fichier versionné qui décrit les modèles employés, pas une copie recopiée dans le notebook ;
- **style appliqué** : chaque image part d'un gabarit qui **nomme le sujet** (un âne gris à quatre pattes, sur ses sabots, longues oreilles, museau allongé, crinière courte) et **écarte explicitement** ce qui a dérivé sans lui — cornes, bois de cervidé, silhouette humaine, bipède — avant d'ajouter la **description visuelle de la scène** en `Scène illustrée : …`. Le gabarit est la constante `GABARIT_ILLUSTRATION` de la cellule suivante ;
- **séparation texte/image** : Qwen-Image rend en lettrage tout texte cité dans le prompt (mesuré le 2026-10-22 : deux lectures visuelles croisées ont vu les paroles dessinées dans l'image) ; la réplique n'est donc **jamais** passée telle quelle au générateur — `decrire_scene_visuelle` la paraphrase en termes visuels via le LLM du kernel, et le plugin ne reçoit que cette description ; la description est en outre **contrainte à l'âne seul** (quatre pattes, quatre sabots au sol, aucun personnage humain, aucun public) — une scène librement décrite dérive vers la posture cabrée ou l'adversaire du duel (round 3 du 2026-10-22), et la scène retenue est affichée avant chaque image ;
- **robustesse** : en cas d'échec (instance injoignable, erreur d'exécution côté serveur), l'exception est journalisée (`logger.error`) et la méthode retourne `None` — le débat continue sans illustration plutôt que de s'arrêter.

L'image reçue est écrite dans un fichier temporaire, et c'est son **chemin** que la méthode retourne : la cellule de duel l'affiche ensuite.


In [7]:
# Plugin de generation d'images : instance ComfyUI de la serie (Qwen-Image 2.1).
import json
import sys
import tempfile
from pathlib import Path


def _charger_aide_comfyui():
    """Importe le client partage de la serie GenAI (shared/helpers)."""
    courant = Path.cwd()
    for _ in range(6):
        aide = courant / "MyIA.AI.Notebooks" / "GenAI" / "shared" / "helpers"
        if aide.is_dir():
            if str(aide) not in sys.path:
                sys.path.insert(0, str(aide))
            break
        courant = courant.parent
    else:
        raise FileNotFoundError(
            "MyIA.AI.Notebooks/GenAI/shared/helpers introuvable depuis " + str(Path.cwd())
        )
    import comfyui_client
    return comfyui_client


comfyui = _charger_aide_comfyui()
client_comfyui = comfyui.load_from_env()
CHEMIN_WORKFLOW = comfyui.dossier_workflows_qwen_image_2_1() / "qwen_image_2_1_text2image.json"
WORKFLOW_IMAGE = comfyui.ui_vers_api(comfyui.charger_workflow_ui(CHEMIN_WORKFLOW))
GRAINE_IMAGE = 12345

# Sujet epingle. Un prompt libre (« style cartoon » + la replique seule) laisse le
# modele choisir le sujet : mesure du 2026-10-22, il a produit un ane a bois de
# cervide puis une silhouette bipede. Le gabarit nomme donc le sujet et ecarte
# explicitement ce qui a derive, AVANT la description de la scene.
GABARIT_ILLUSTRATION = (
    "Style cartoon comique, dessin animé pour enfants. "
    "Un âne gris vu de profil, strictement quadrupède : les quatre sabots "
    "posés au sol et les quatre pattes portant le poids, longues oreilles, "
    "museau allongé, crinière courte. Aucune pose cabrée, aucun appui sur "
    "les pattes arrière seules, aucune corne, aucun bois de cervidé, "
    "aucune silhouette humaine, pas de bipède. Aucun texte, aucune "
    "inscription, aucun lettrage dans l'image : la réplique ne doit pas "
    "y être écrite. "
    "Scène illustrée : {scene}"
)


# Separation texte/image : Qwen-Image rend en lettrage tout texte cite
# dans le prompt. Mesure du 2026-10-22 (deux QA visuelles croisees) : les
# paroles passees telles quelles etaient dessinees ; la scene est donc
# paraphrasee visuellement par le LLM du kernel, sans recopier la replique.
PROMPT_SCENE = (
    "Tu prépares une illustration où NE FIGURE QUE L'ÂNE. Décris en UNE "
    "phrase courte, uniquement en termes visuels, ce que fait l'Âne dans "
    "la réplique ci-dessous. Contraintes : l'âne est le seul personnage "
    "(aucun humain, aucune silhouette debout, aucun public, aucun autre "
    "animal), strictement à quatre pattes, les quatre sabots au sol, "
    "aucune pose cabrée ni redressée ; seuls l'âne et le décor (sol, "
    "rideau, lumière) apparaissent. Ne recopie aucun mot de la réplique : "
    "aucune citation. Réponds en français, en une seule phrase, sans "
    "guillemets.\n\n"
    "Réplique :\n{replique}"
)


async def decrire_scene_visuelle(kernel, replique):
    """Renvoie une description visuelle de la scène, sans aucun texte
    recopié de la réplique — le générateur dessine le texte qu'on cite.
    """
    resultat = await kernel.invoke_prompt(
        prompt=PROMPT_SCENE.format(replique=replique)
    )
    valeur = getattr(resultat, "value", resultat)
    if isinstance(valeur, list):
        valeur = "".join(getattr(m, "content", str(m)) or "" for m in valeur)
    scene = " ".join(str(valeur).split())
    return scene or "l'âne gris, seul, à quatre pattes, chante sur une scène de théâtre vide"


class ImagePlugin:
    def __init__(self, client, workflow, graine=GRAINE_IMAGE):
        self.client = client
        self.workflow = workflow
        self.graine = graine

    @kernel_function(name="generate_image", description="Genere une illustration via l'instance ComfyUI de la serie ; context = description visuelle de la scene, jamais la replique mot a mot")
    async def generate(self, context: str) -> str:
        try:
            # Copie : chaque replique part du graphe versionne, sans le modifier.
            prompt_api = json.loads(json.dumps(self.workflow))
            comfyui.noeud_de_classe(prompt_api, "CLIPTextEncode")["inputs"]["text"] = (
                GABARIT_ILLUSTRATION.format(scene=context)
            )
            comfyui.noeud_de_classe(prompt_api, "KSampler")["inputs"]["seed"] = self.graine
            prompt_id = self.client.queue_prompt(prompt_api)
            # 1800 s : une generation seule mesuree a 568 s le 2026-10-22,
            # et l'instance est partagee -- une requete en file depasse 900 s.
            resultat = self.client.wait_for_completion(prompt_id, timeout=1800)
            images = [im for sortie in resultat.get("outputs", {}).values()
                      for im in sortie.get("images", [])]
            if not images:
                logger.error("Instance ComfyUI : aucune image renvoyee")
                return None
            donnee = self.client.get_output_file(
                images[0]["filename"], images[0].get("subfolder", ""), images[0].get("type", "output")
            )
            chemin = Path(tempfile.gettempdir()) / f"barbie_ane_{prompt_id}.png"
            chemin.write_bytes(donnee)
            return str(chemin)
        except Exception as e:
            logger.error(f"Erreur generation image: {(str(e).splitlines() or [repr(e)])[0]}")
            return None


print("Classe ImagePlugin définie (instance ComfyUI)")


Classe ImagePlugin définie (instance ComfyUI)


## Configuration des agents

Chaque duelliste est un `ChatCompletionAgent` avec son **propre kernel** (`kernel_barbie`, `kernel_ane`) et des instructions système distinctes :

- **Barbie** : « Défends des positions optimistes avec élégance » — le ton soigné ;
- **L'Âne de Shrek** : « Utilise l'humour absurde et décalé » — le ton débridé.

Les deux instructions embarquent la contrainte linguistique choisie (`contrainte_choisie[1]`), interpolée dans le prompt système : c'est ce qui garantit que **chaque** réplique, quel que soit l'agent, respecte la règle du débat.

Point de câblage important : le plugin `ImagePlugin` est ajouté **uniquement au kernel de l'Âne** (`kernel_ane.add_plugin(...)`). Barbie parle, l'Âne parle *et* illustre — la dissymétrie est volontaire et montre que les plugins sont attachés par kernel, pas globalement.

In [8]:
# Création des kernels séparés
kernel_barbie = create_kernel()
kernel_ane = create_kernel()

image_Plugin = ImagePlugin(client_comfyui, WORKFLOW_IMAGE)
# Ajout du plugin uniquement à l'Âne
kernel_ane.add_plugin(image_Plugin, plugin_name="image_gen")


barbie = ChatCompletionAgent(
    kernel=kernel_barbie,
    name="Barbie",
    instructions=f"""
    Tu es Barbie. Défends des positions optimistes avec élégance.
    Contrainte obligatoire : {contrainte_choisie[1]}
    """
)

ane = ChatCompletionAgent(
    kernel=kernel_ane,
    name="Ane_Shrek",
    instructions=f"""
    Tu es l'Âne de Shrek. Utilise l'humour absurde et décalé.
    Contrainte obligatoire : {contrainte_choisie[1]}
    """
)

print("Kernels et agents Barbie/Âne créés")


Kernels et agents Barbie/Âne créés


### Exemple guidé 2 — Création d'un troisième agent

*Contribution étudiante du groupe de la PR #21089, intégrée comme exemple guidé.*

Le débat oppose Barbie et l'Âne. L'exemple ajoute un **troisième personnage** : un **Arbitre** qui commente les performances des deux duellistes.

**Ce que fait le code ci-dessous** :

- `# Étape 1` — créer un kernel dédié à l'Arbitre avec `create_kernel()` ;
- `# Étape 2` — définir ses instructions système (juger le respect de la contrainte, la créativité, l'humour, terminer par une note sur 10) ;
- l'agent `arbitre` est ensuite instancié comme les deux autres.

L'instruction de l'énoncé d'origine est conservée : s'inspirer des instructions de Barbie et de l'Âne pour le format du prompt.


In [9]:
# Exemple guidé 2 — Création d'un troisième agent (Arbitre).
# Contribution étudiante du groupe de la PR #21089, intégrée comme exemple guidé.

# Etape 1 : un kernel dedie, comme pour Barbie et l'Ane
kernel_arbitre = create_kernel()

# Etape 2 : les instructions systeme de l'arbitre
ARBITRE_INSTRUCTIONS = """
Tu es l'Arbitre du duel verbal entre Barbie et l'Âne de Shrek.
À chaque tour, commente en 2 phrases maximum la dernière réplique :
  1. le respect de la contrainte linguistique (rime, style imposé...) ;
  2. la créativité et l'humour.
Termine par une note sur 10. Reste neutre et impartial.
Contrainte obligatoire pour tes propres commentaires : %s
""" % contrainte_choisie[1]

# Creation de l'agent
arbitre = ChatCompletionAgent(
    kernel=kernel_arbitre,
    name="Arbitre",
    instructions=ARBITRE_INSTRUCTIONS,
)

print(f"Agent créé : {arbitre.name}")
# Pour l'utiliser dans le duel : AgentGroupChat(agents=[barbie, ane, arbitre], ...)


Agent créé : Arbitre


### Lecture du résultat — exemple guidé 2

L'agent `Arbitre` est créé avec son propre kernel et ses instructions, et l'appel `print` affiche bien `Agent créé : Arbitre`. La construction demandée par l'énoncé est donc complète.

**Limite de l'exemple, et elle est de taille : l'Arbitre ne participe à aucun duel.** Le commentaire de la dernière ligne le dit — `# Pour l'utiliser dans le duel : AgentGroupChat(agents=[barbie, ane, arbitre], ...)` — mais la cellule de duel plus bas construit toujours `AgentGroupChat(agents=[barbie, ane])`. Autrement dit, l'Arbitre existe, il n'est jamais convoqué : le carnet ne montre nulle part ses commentaires ni ses notes. Un agent créé mais absent de la boucle ne produit rien — c'est exactement ce que l'exercice 2 de remplacement demande de corriger.


## Stratégie de terminaison

Sans règle d'arrêt, un `AgentGroupChat` tournerait indéfiniment. `DebateTerminationStrategy` sous-classe `TerminationStrategy` : le débat s'arrête dès que l'historique atteint `MAX_TURNS = 5` messages.

Trois détails d'implémentation à noter :

- **`ClassVar[int]`** : `MAX_TURNS` est une constante de classe, pas un champ d'instance — l'annotation `ClassVar` évite le conflit avec le mécanisme d'annotations de Semantic Kernel ;
- **le point d'accroche** : dans Semantic Kernel, la méthode que l'orchestrateur appelle est `should_terminate`, qui vérifie la portée de la stratégie puis **délègue à `should_agent_terminate`** — c'est cette seconde méthode que la documentation désigne comme celle à implémenter dans une sous-classe. Surcharger `should_terminate` fonctionne (l'orchestrateur l'appelle bien), mais court-circuite la vérification de portée intégrée ;
- **le contrat `async`** : `should_agent_terminate` est `async`, ce qui permet une terminaison dynamique — interroger le contenu du dernier message, un service externe, etc. C'est la piste de l'exemple guidé 3.

La stratégie est passée à l'`AgentGroupChat` au moment de sa construction, et est consultée après chaque message ajouté à l'historique.


In [10]:
from typing import ClassVar
from semantic_kernel.agents.strategies.termination.termination_strategy import TerminationStrategy

class DebateTerminationStrategy(TerminationStrategy):
    MAX_TURNS: ClassVar[int] = 5  # Annotation correcte avec ClassVar

    async def should_agent_terminate(self, agent, history) -> bool:
        return len(history) >= self.MAX_TURNS

print("Stratégie de terminaison du débat configurée")


Stratégie de terminaison du débat configurée


## Lancement du débat

L'`AgentGroupChat` rassemble les deux agents (`barbie`, `ane`) et la stratégie de terminaison définie plus haut. L'appel `add_chat_message` amorce la conversation — Semantic Kernel exige un historique non vide — puis `group_chat.invoke()` fait tourner le duel :

- chaque agent produit sa réplique à tour de rôle, conformément à la stratégie de sélection séquentielle ;
- **spécificité pédagogique** : à chaque réplique de l'Âne, le code fait d'abord paraphraser la réplique en **description visuelle** par `decrire_scene_visuelle`, contrainte à l'âne seul à quatre pattes (une scène librement décrite dérive vers la posture cabrée ou l'adversaire du duel) — le générateur rend en lettrage tout texte cité dans le prompt, la réplique ne lui est donc jamais passée telle quelle — puis invoque le plugin `image_gen` (`generate_image`) avec cette description en contexte et affiche l'image renvoyée par l'instance ComfyUI (`width=320`) ;
- les répliques sont aussi conservées dans `repliques_du_duel` : les exercices de la section 9 les réutilisent au lieu de relancer un duel.

C'est ici que se matérialise la boucle *parler puis illustrer* : la réplique d'un agent est d'abord résumée en scène visuelle, qui devient l'argument de l'appel d'image, reliant le service de conversation et l'instance ComfyUI dans le même débat. La génération d'une illustration prend **plusieurs minutes** — le chargement des modèles puis les étapes de diffusion de ComfyUI dominent largement le reste ; c'est de loin le poste le plus lent du montage.


In [11]:
repliques_du_duel = []

async def run_debate():
    logger.info(f"Contrainte active : {contrainte_choisie[0]}")

    group_chat = AgentGroupChat(
        agents=[barbie, ane],
        termination_strategy=DebateTerminationStrategy()
    )

    # Add initial topic message (SK requires non-empty chat history)
    await group_chat.add_chat_message(
        ChatMessageContent(role=AuthorRole.USER, content="Commencez le duel verbal !")
    )

    async for msg in group_chat.invoke():
        repliques_du_duel.append(msg)
        print(f"\n{msg.name}: {msg.content}")

        if msg.name == "Ane_Shrek":
            try:
                # Separation replique/image : le generateur rend en lettrage
                # tout texte cite dans le prompt (mesure du 2026-10-22) ; la
                # scene est decrite visuellement, sans recopier la replique.
                scene = await decrire_scene_visuelle(ane.kernel, msg.content)
                print(f"Scène visuelle envoyée au générateur : {scene}")
                image_result = await ane.kernel.invoke(
                    function_name="generate_image",
                    plugin_name="image_gen",
                    arguments=KernelArguments(context=scene)
                )
                chemin = str(image_result) if image_result else ""
                if chemin and os.path.exists(chemin):
                    display(IPImage(filename=chemin, width=320))
                else:
                    logger.warning("Illustration indisponible pour cette replique")
            except Exception as e:
                logger.warning(f"Image generation skipped: {e}")

await run_debate()


INFO:BarbieVsAne:Contrainte active : Rime


INFO:semantic_kernel.agents.group_chat.agent_chat:Adding `1` agent chat messages


INFO:semantic_kernel.agents.strategies.selection.sequential_selection_strategy:Selected agent at index 0 (ID: 7a1a48b8-c3fc-4a6d-8ee2-94ce8ad36d54, name: Barbie)


INFO:semantic_kernel.agents.group_chat.agent_chat:Invoking agent Barbie


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


INFO:semantic_kernel.connectors.ai.open_ai.services.open_ai_handler:OpenAI usage: CompletionUsage(completion_tokens=26, prompt_tokens=54, total_tokens=80, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


... (40 outputs left out)

Scène visuelle envoyée au générateur : Un âne tropical à quatre pattes, les quatre sabots bien ancrés au sol, braie gueule ouverte sous une lumière chaude devant un rideau de scène, dans une atmosphère de chaleur humide.


ERROR:BarbieVsAne:Erreur generation image: Workflow execution failed (server reported error): [


INFO:semantic_kernel.functions.kernel_function:Function image_gen-generate_image succeeded.


INFO:semantic_kernel.functions.kernel_function:Function completed. Duration: 34.612248s


### Lecture du résultat : le duel verbal réel

L'exécution montre le protocole complet en action, tour par tour :

- **Sélection séquentielle des agents** : les logs `Selected agent at index 0/1` alternent Barbie puis l'Âne, conformément à `AgentGroupChat` et à sa stratégie de sélection par défaut ;
- **Contrainte respectée** : la contrainte tirée au sort est portée par les instructions système des deux agents, et leurs répliques la suivent ;
- **Le plugin d'image n'est branché que sur l'Âne** : seul `Ane_Shrek` déclenche `image_gen-generate_image`, conformément au câblage `kernel_ane.add_plugin(...)` ;
- **La terminaison fonctionne** : dès que l'historique atteint `MAX_TURNS`, `DebateTerminationStrategy` coupe la boucle.

La sortie est la **preuve de bout en bout** du montage : deux agents contraints, un plugin d'image branché sur l'instance ComfyUI de la série, une stratégie de terminaison — le tout orchestré par Semantic Kernel.


### Exemple guidé 3 — Stratégie de terminaison personnalisée

*Contribution étudiante du groupe de la PR #21089, intégrée comme exemple guidé.*

L'implémentation de base arrête le débat après un nombre fixe de tours (`MAX_TURNS = 5`). L'exemple vise un arrêt **sur le contenu** : couper le débat dès qu'un personnage capitule.

**Ce que fait le code ci-dessous** :

- `# Étape 1` — définir la liste des mots-clés déclencheurs (`TRIGGER_WORDS`) — des radicaux, pour couvrir « capitule », « capituler », « capitulation » ;
- `# Étape 2` — lire le dernier message (`history[-1]`), le mettre en minuscules, et déclencher l'arrêt si l'un des mots-clés y apparaît ;
- un garde-fou `MAX_TURNS = 10` arrête le débat même si personne ne cède.

L'indice de l'énoncé d'origine est conservé : utiliser `str(history[-1].content).lower()` pour lire le dernier message.


In [12]:
# Exemple guidé 3 — Stratégie de terminaison personnalisée (capitulation).
# Contribution étudiante du groupe de la PR #21089, intégrée comme exemple guidé.
from typing import ClassVar
from semantic_kernel.agents.strategies.termination.termination_strategy import TerminationStrategy

class SurrenderTerminationStrategy(TerminationStrategy):
    # Etape 1 : mots-cles declencheurs (radicaux, pour couvrir capitule / capituler / capitulation...)
    TRIGGER_WORDS: ClassVar[list[str]] = ["capitul", "abandon", "je me rends", "j'abandonne"]
    MAX_TURNS: ClassVar[int] = 10  # garde-fou : le debat s'arrete meme si personne ne cede

    async def should_terminate(self, agent, history, cancellation_token=None) -> bool:
        if not history:
            return False
        # Etape 2 : verifier si le dernier message contient un mot-cle
        dernier = str(history[-1].content).lower()
        result = any(mot in dernier for mot in self.TRIGGER_WORDS) or len(history) >= self.MAX_TURNS
        return result

print("SurrenderTerminationStrategy définie")
# Utilisation : AgentGroupChat(agents=[barbie, ane], termination_strategy=SurrenderTerminationStrategy())


SurrenderTerminationStrategy définie


In [13]:
# Exemple guidé 3 — la stratégie en action : un duel court, sans illustration.
# Le duel ci-dessous n'appelle pas le plugin d'image : on ne mesure ici que la terminaison.

async def duel_capitulation():
    chat = AgentGroupChat(
        agents=[barbie, ane],
        termination_strategy=SurrenderTerminationStrategy()
    )
    await chat.add_chat_message(
        ChatMessageContent(role=AuthorRole.USER, content="Commencez le duel verbal !")
    )
    repliques = []
    async for msg in chat.invoke():
        repliques.append(msg)
        print(f"\n{msg.name}: {msg.content}")
    return repliques


repliques_capitulation = await duel_capitulation()
dernier = str(repliques_capitulation[-1].content).lower() if repliques_capitulation else ""
print(f"\n{len(repliques_capitulation)} répliques ; un mot déclencheur apparaît dans la dernière : "
      f"{any(mot in dernier for mot in SurrenderTerminationStrategy.TRIGGER_WORDS)}")


INFO:semantic_kernel.agents.group_chat.agent_chat:Adding `1` agent chat messages


INFO:semantic_kernel.agents.strategies.selection.sequential_selection_strategy:Selected agent at index 0 (ID: 7a1a48b8-c3fc-4a6d-8ee2-94ce8ad36d54, name: Barbie)


INFO:semantic_kernel.agents.group_chat.agent_chat:Invoking agent Barbie


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


INFO:semantic_kernel.connectors.ai.open_ai.services.open_ai_handler:OpenAI usage: CompletionUsage(completion_tokens=33, prompt_tokens=54, total_tokens=87, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


INFO:semantic_kernel.agents.strategies.selection.sequential_selection_strategy:Selected agent at index 1 (ID: b410afca-4b04-4e79-83fd-9adfe6179635, name: Ane_Shrek)


... (36 outputs left out)


Ane_Shrek: Je claque des sabots, je lance un rire **bestial**, viens donc m’chercher ici, duel 100% **verbal**.


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


INFO:semantic_kernel.connectors.ai.open_ai.services.open_ai_handler:OpenAI usage: CompletionUsage(completion_tokens=34, prompt_tokens=670, total_tokens=704, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))



Barbie: Je lance le défi, sourire clair et **royal** : que ton prochain coup soit **verbal**, et notre échange **fraternal**.

9 répliques ; un mot déclencheur apparaît dans la dernière : False


### Lecture du résultat — exemple guidé 3

**Ce qui marche.** La stratégie lit bien le dernier message, et l'arrêt par mots-clés fonctionne : quand le duel se termine sans capitulation, c'est le garde-fou `MAX_TURNS = 10` qui coupe, et le compte de répliques le montre. La stratégie est donc **exécutée**, contrairement à sa première version restée non exécutée dans la copie de travail du groupe.

**Deux limites à nommer.**

1. **La détection est une recherche de sous-chaîne** (`mot in dernier`). Elle ne distingue ni la personne, ni la négation : « je n'abandonnerai jamais » ou « tu vas abandonner, l'Âne » contiennent tous deux un déclencheur, et coupent le débat alors que personne ne cède. La stratégie détecte donc *une occurrence*, pas *une capitulation*.
2. **La méthode surchargée n'est pas le point d'accroche canonique.** La classe redéfinit `should_terminate`, que l'orchestrateur appelle — cela marche — mais Semantic Kernel délègue de `should_terminate` vers `should_agent_terminate`, la méthode prévue pour les sous-classes. Surcharger `should_terminate` court-circuite la vérification de portée (`agents`) intégrée. La cellule de la section « Stratégie de terminaison » illustre la forme canonique.

L'exercice 3 de remplacement demande de traiter ces deux points.


## 9. Exercices

Trois exercices de remplacement, sur le même carnet. Chacun prolonge l'un des trois exemples guidés — même socle, question différente. Les cellules ci-dessous sont des **squelettes** : elles s'exécutent telles quelles et affichent ce qu'il reste à faire.

| Exercice | Prolonge | Question |
|---|---|---|
| 1 | Exemple guidé 1 (contraintes) | rendre une contrainte **vérifiable par le code** et mesurer son taux de respect |
| 2 | Exemple guidé 2 (Arbitre) | faire **entrer l'Arbitre dans le duel** et exploiter ses notes |
| 3 | Exemple guidé 3 (terminaison) | détecter une capitulation **sans se tromper de cible** |


### Exercice 1 — Une contrainte que le code peut vérifier

L'exemple guidé 1 ajoute des contraintes **déclaratives** : leur description est une phrase en français, et rien ne vérifie que le modèle l'a suivie. L'objectif est de rendre une contrainte **mesurable**.

**Objectif** :

1. définir une règle vérifiable mécaniquement — par exemple « la réplique contient au moins un mot de la liste `MOTS_ANES` » ;
2. écrire `taux_de_respect(repliques, regle)` qui reçoit les répliques d'un duel et **retourne la proportion** de répliques qui respectent la règle ;
3. l'appliquer au duel déjà présent dans le notebook (variable `repliques_du_duel`).

**Indices** :

- `# Étape 1` : une contrainte vérifiable = un nom, une description **et** une fonction `regle(texte) -> bool` ;
- `# Étape 2` : `result = None  # TODO étudiant`, puis le calcul réplique par réplique ;
- `# Indice` : `sum(1 for r in repliques if regle(str(r))) / len(repliques)`.


In [14]:
# Exercice 1 — Une contrainte que le code peut vérifier.
# L'expérience est autonome : elle ne modifie ni CONTRAINTES, ni les agents, ni le duel.


def taux_de_respect(repliques, regle):
    # TODO étudiant : retourner la proportion de repliques qui respectent `regle`.
    # Étape 1 : compter les repliques ou `regle(str(replique))` est vrai.
    # Étape 2 : diviser par le nombre de repliques (attention au cas vide).
    # Indice : sum(1 for r in repliques if regle(str(r))) / len(repliques)
    result = None  # TODO étudiant
    return result


taux = taux_de_respect(repliques_du_duel, lambda texte: "ane" in texte.lower() or "âne" in texte.lower())
if taux is None:
    print("Exercice 1 à compléter : le taux de respect d'une contrainte vérifiable par le code "
          f"({len(repliques_du_duel)} répliques disponibles dans repliques_du_duel).")
else:
    print(f"Taux de respect : {taux:.0%} sur {len(repliques_du_duel)} répliques")


Exercice 1 à compléter : le taux de respect d'une contrainte vérifiable par le code (4 répliques disponibles dans repliques_du_duel).


### Exercice 2 — Mettre l'Arbitre dans le duel

L'exemple guidé 2 crée l'agent `Arbitre`, mais le duel tourne toujours à deux : `AgentGroupChat(agents=[barbie, ane])` ne le convoque jamais. L'objectif est de le faire entrer dans la boucle **et** d'exploiter ce qu'il produit.

**Objectif** :

1. relancer un duel avec `agents=[barbie, ane, arbitre]` — pensez à relever `MAX_TURNS`, sinon l'Arbitre parle au mieux une fois ;
2. collecter les répliques de ce nouveau duel ;
3. écrire `notes_arbitre(repliques)` qui retourne la liste des notes sur 10 que l'Arbitre a attribuées, extraites du texte (par exemple avec `re.compile(r"(\d+(?:[.,]\d+)?)\s*/\s*10")`) ;
4. afficher le nombre de notes et leur moyenne.

**Indices** :

- `# Étape 1` : `chat = AgentGroupChat(agents=[barbie, ane, arbitre], termination_strategy=DebateTerminationStrategy())` ;
- `# Étape 2` : parcourir les répliques dont `msg.name == "Arbitre"` ;
- `# Indice` : `float(m.group(1).replace(",", "."))` pour convertir la note trouvée.


In [15]:
# Exercice 2 — Mettre l'Arbitre dans le duel et lire ses notes.
# `arbitre` a été créé dans l'exemple guidé 2 ; `repliques_du_duel` vient du duel de la section « Lancement du débat ».
import re


async def duel_avec_arbitre(max_tours=12):
    # TODO étudiant : relancer un duel a trois agents et retourner ses repliques.
    # Étape 1 : construire AgentGroupChat(agents=[barbie, ane, arbitre], ...) avec une
    #           strategie de terminaison dont MAX_TURNS vaut `max_tours`.
    # Étape 2 : amorcer l'historique puis collecter les messages dans une liste.
    result = None  # TODO étudiant
    return result


def notes_arbitre(repliques):
    # TODO étudiant : extraire les notes sur 10 des repliques de l'Arbitre.
    # Indice : re.compile(r"(\d+(?:[.,]\d+)?)\s*/\s*10") sur str(replique.content).
    result = None  # TODO étudiant
    return result


repliques_arbitre = await duel_avec_arbitre()
if repliques_arbitre is None:
    print("Exercice 2 à compléter : convier l'Arbitre au duel, puis extraire ses notes sur 10.")
else:
    notes = notes_arbitre(repliques_arbitre)
    if not notes:
        print("Exercice 2 à compléter : aucune note extraite des repliques de l'Arbitre.")
    else:
        print(f"{len(notes)} notes d'arbitre, moyenne {sum(notes) / len(notes):.2f}/10")


Exercice 2 à compléter : convier l'Arbitre au duel, puis extraire ses notes sur 10.


### Exercice 3 — Une détection de capitulation qui ne se trompe pas de cible

L'exemple guidé 3 détecte la capitulation par **sous-chaîne** dans le dernier message. Deux façons de se tromper, toutes deux mesurables : « je n'abandonnerai jamais » (négation) et « tu vas abandonner, l'Âne » (parler de l'autre) déclenchent l'arrêt alors que personne ne cède.

**Objectif** :

1. écrire `capitule(phrase)` qui ne renvoie `True` que pour une capitulation **du locuteur** — à la première personne et non niée ;
2. la tester sur une liste de phrases pièges, **sans appeler le modèle** : c'est un test unitaire, il doit être déterministe ;
3. (bonus) réimplémenter `SurrenderTerminationStrategy` sur le point d'accroche canonique `should_agent_terminate` et vérifier que le duel se comporte pareil.

**Indices** :

- `# Étape 1` : `re.compile(r"\b(j'abandonne|je me rends|je capitule)\b")` — et traiter la négation (`n'`, `ne ... jamais`, `pas`) ;
- `# Étape 2` : tester au minimum « Je n'abandonnerai jamais ! », « Tu vas abandonner, l'Âne ! », « J'abandonne, tu gagnes. » ;
- `# Indice` : retourner un `dict` phrase -> booléen attendu/observé, et compter les écarts.


In [16]:
# Exercice 3 — Une detection de capitulation qui ne se trompe pas de cible.
# Test unitaire : aucune replique n'est generee ici, tout est deterministe.
import re

PHRASES_PIEGES = [
    ("Je n'abandonnerai jamais !", False),
    ("Tu vas abandonner, l'Âne !", False),
    ("J'abandonne, tu gagnes.", True),
]


def capitule(phrase):
    # TODO étudiant : True seulement si le LOCUTEUR annonce sa propre capitulation.
    # Étape 1 : reperer les formes a la premiere personne (j'abandonne, je me rends, je capitule).
    # Étape 2 : exclure les formes niees (n'... pas / jamais) et les formes ou l'autre est vise.
    result = None  # TODO étudiant
    return result


verdicts = {phrase: capitule(phrase) for phrase, attendu in PHRASES_PIEGES} if capitule("test") is not None else None
if verdicts is None:
    print("Exercice 3 à compléter : une détection de capitulation par locuteur, testée sur "
          f"{len(PHRASES_PIEGES)} phrases pièges.")
else:
    ecarts = [p for p, attendu in PHRASES_PIEGES if verdicts.get(p) != attendu]
    print(f"{len(PHRASES_PIEGES) - len(ecarts)}/{len(PHRASES_PIEGES)} phrases classées correctement")
    for phrase in ecarts:
        print(f"  écart : {phrase!r} -> {verdicts.get(phrase)}")


Exercice 3 à compléter : une détection de capitulation par locuteur, testée sur 3 phrases pièges.


## Conclusion

Ce notebook a démontré un **débat multi-agents contraint** avec Semantic Kernel : deux `ChatCompletionAgent` (Barbie et l'Âne de Shrek) s'affrontent en respectant une contrainte linguistique tirée au sort, et l'Âne illustre ses répliques par des images générées sur l'instance ComfyUI de la série.

**Points clés** :

- **le protocole de sélection séquentielle** alterne les agents via `SequentialSelectionStrategy`, les logs `Selected agent at index N` traçant chaque tour ;
- **la contrainte linguistique est portée par l'instruction système** : c'est elle qui contraint la génération, agent par agent ;
- **la génération d'images passe par un service externe** (ComfyUI), appelé en HTTP avec le workflow versionné de la série — le service de conversation et le service d'images sont deux dépendances distinctes ;
- **la stratégie de terminaison fonctionne** : le débat s'arrête selon la règle implémentée.

**Les exercices de la section « 9. Exercices »** prolongent les exemples guidés 1, 2 et 3 — une contrainte vérifiable par le code, l'Arbitre réellement convoqué dans le duel, et une détection de capitulation qui ne se trompe pas de cible. L'architecture (kernel + plugin + stratégie de terminaison) est réutilisable telle quelle pour d'autres formats de joute (débat politique, concours de poésie, improvisation théâtrale).
